# Fitting for a point source polarization with `LeakageLib`

This jupyter notebook fits for the polarization of a single point source, with a photon and particle background. Spatial, particle, and spectral weights are used.

In [1]:
import leakagelib

>>> PyXSPEC is not installed, you will no be able to use it.


First, we need to load the data. `LeakageLib` needs the unzipped level 2 (event_l2) event files, the housekeeping (hk) attitude files, and optionally the auxiliary (auxil) exposure maps in order to run. (The attitude file contains the spacecraft orientation which is necessary to get the PSF model right). You can download these files from the IXPE archive.

I made mock IXPE data for this example, which we'll use.

In [2]:
datas = leakagelib.IXPEData.load_all_detectors_with_path("data", "ps")

>>> Reading (in memory) /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...


<div class="alert alert-info">

**Alternatives to** `IXPEData.load_all_detectors_with_path`

- `IXPEData.load_all_detectors`. Searches all directories listed in the DATA_DIRECTORIES variable defined in settings.py for the given obs id, and load all detectors.
- `IXPEData.load_all_detectors_with_path`. Provide a path to the observation and the function will load all detectors. This will not use the DATA_DIRECTORIES variable.
- `IXPEData`. You must pass specific file names.
</div>

Next we center the data and cut to 2-8 keV.

In [3]:
for data in datas:
    data.iterative_centroid_center()
    data.retain(data.evt_energies > 2)
    data.retain(data.evt_energies < 8)

<div class="alert alert-info">

**Alternatives to** `IXPEData.iterative_centroid_center`

- `IXPEData.explicit_center`: Center on a specific point, in pixels.
- `IXPEData.iterative_centroid_center`: Center by iteratively zooming in on the average event. This finds central PSs rather well.
</div>

Now we have to tell the fitter how we want to do the fit. First, we tell it to only consider events within 280 arcseconds of the center. More distant events will automatically be removed from the data sets.

In [4]:
settings = leakagelib.FitSettings(datas)
settings.apply_circular_roi(280)

6478 events were cut for being outside the region of interest.


<div class="alert alert-info">

**Alternatives to** `FitSettings.apply_circular_roi`

- `IXPEData.apply_circular_roi`: Assumes a circular ROI centered on the origin
- `IXPEData.apply_roi`: Apply an arbitrary ROI, which you pass in as an image. The image must have the same pixel size and dimensions as the fit sources.
</div>

You might be curious how many events are left. You can check this here:

In [5]:
sum([len(data) for data in datas])

15692

Now we have to tell it the number of components in the fit. For this simplified scenario there are two components: a point source and a photon background. To create the point source,

In [6]:
settings.add_point_source() # Named "src" by default
settings.fix_flux("src", 1)
settings.set_initial_qu("src", (0.5,0))

Without loss of generality, we have defined the flux to be in units of the source flux.

<div class="alert alert-info">

You can use `FitSettings.set_initial_qu` to set the start point of the fitter. It's important to set the start point as close to the true value as possible to avoid falling into the wrong maximum of the likelihood.

</div>

Now we create the background models. These background sources are assumed to be spatially uniform and already have spatial weights applied. 

In [7]:
settings.add_background() # Named "bkg" by default
settings.fix_qu("bkg", (0, 0)) # Assume an unpolarized background
settings.add_particle_background() # Named "pbkg" by default
settings.fix_qu("pbkg", (0, 0)) # Assume the particles have no direction dependence

To use energy weights, we must assign a spectrum. To do this, we create functions which take in the energy in keV and return the photon spectrum (dN/dE). Normalization doesn't matter. The ARF and RMF are automatically included.

LeakageLib employs some approximations when using spectral weights. These approximations would not suitable for _spectral_ fitting or simultaneous spectral and polarimetric fitting. But for this application (polarimetric fitting alone), the approximations are suitable.

In [8]:
settings.set_spectrum("bkg", lambda e: e**-2.5) # Gamma=2.5, unabsorbed powerlaw
settings.set_spectrum("src", lambda e: e**-1.5) # Gamma=1.5, unabsorbed powerlaw

>>> Reading (in memory) /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/gpd/cpf/arf/ixpe_d1_obssim20240101_v013.arf...
>>> Using cached xVignetting object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...
>>> Using cached xEffectiveArea object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/gpd/cpf/arf/ixpe_d1_obssim20240101_v013.arf...
>>> Using cached xVignetting object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...


We have assigned all the weights. Now we are ready to create the fitter...

In [9]:
fitter = leakagelib.Fitter(settings)

Data set ps DU 1 had no exposure map loaded. Please load an exposure map if you are fitting to events in the vignetted portion.
>>> Using cached xVignetting object at /opt/homebrew/anaconda3/lib/python3.12/site-packages/ixpeobssim/caldb/ixpe/xrt/bcf/vign/ixpe_d1_obssim20240101_vign_v013.fits...


... and run the fit

In [10]:
result = fitter.fit()
result

FitResult:
	q (src) = 0.5160 +/- 0.0726
	u (src) = 0.0007 +/- 0.0729
	f (bkg) = 1.5002 +/- 0.0323
	f (pbkg) = 0.6969 +/- 0.0180

Polarization:
	PD (src): 0.5160 +/- 0.0726
	PA (src): 0.0405 deg +/- 4.0493
Likelihood 22081.859047182963, dof 15688
Optimization terminated successfully.

Indeed, the true polarization of this mock source was q = 0.5, u = 0.0. The fit result agrees.

<div class="alert alert-info">

**Alternatives to** `Fitter.fit`

- `Fitter.fit`: Fit by numerically maximizing the likelihood and get Gaussian uncertainties by taking the Hessian and using Laplace's approximation.
- `IXPEData.fit_mcmc`: Do a full MCMC fit.

</div>

You can access `result.params` and `result.sigmas` to get the best-fit values and uncertainties. Also `result.cov` to get the full covariance matrix. The covariance matrix entries are stored in the same order as the `result.parameter_names` array.

In [13]:
print("Parameters", result.params)
print("Uncertainties", result.sigmas)
print()
print("Parameters", result.parameter_names)
print("Covariance", result.cov)

Parameters {('q', 'src'): 0.5159648137667769, ('u', 'src'): 0.0007295289926836142, ('f', 'bkg'): 1.500223629181317, ('f', 'pbkg'): 0.696922379858947}
Uncertainties {('q', 'src'): 0.07264390786728002, ('u', 'src'): 0.07292986443927577, ('f', 'bkg'): 0.03232077283043227, ('f', 'pbkg'): 0.018018473699039077}

Parameters [('q', 'src'), ('u', 'src'), ('f', 'bkg'), ('f', 'pbkg')]
Covariance [[ 5.27713735e-03 -8.61079139e-05  0.00000000e+00  0.00000000e+00]
 [-8.61079139e-05  5.31876513e-03  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  1.04463236e-03  2.08233547e-04]
 [ 0.00000000e+00  0.00000000e+00  2.08233547e-04  3.24665394e-04]]
